# Earth Intelligence Agent — Starter Kit
### TCC Track | UK AI Agent Hack Ep.4 | March 2026

This notebook gets you started with satellite data and shows basic analytics you can build on.

**What you'll learn:**
1. How to search for and load Sentinel-2 satellite imagery
2. Basic analytics: true color display, NDVI, cloud masking
3. Change detection between two dates
4. Connecting real-world events to satellite data

**Run this in Google Colab for the easiest setup.**

In [ ]:
# Run this cell first — installs everything you need
import subprocess, sys
for pkg in ['pystac-client', 'planetary-computer', 'rasterio', 'rioxarray', 'matplotlib', 'numpy', 'requests']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print("All packages installed!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pystac_client import Client
import planetary_computer
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds
import requests
import warnings
warnings.filterwarnings('ignore')

# Make all plots big by default
plt.rcParams['figure.dpi'] = 120

# Connect to Microsoft Planetary Computer (free, no API key needed)
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print("Connected to Planetary Computer")

## 1. Finding Satellite Data

Sentinel-2 captures the entire Earth every 5 days at 10m resolution with 13 spectral bands. We'll search for cloud-free imagery over any location.

In [ ]:
# Search for Sentinel-2 data over Imperial College London
# Change these coordinates to search anywhere on Earth
bbox = [-0.195, 51.490, -0.165, 51.510]  # [west, south, east, north]
location_name = "Imperial College London"

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-06-01/2025-09-01",  # Summer 2025 — less clouds
    query={"eo:cloud_cover": {"lt": 10}},  # Less than 10% cloud cover
    max_items=5,
)

items = list(search.items())
print(f"Found {len(items)} cloud-free images over {location_name}\n")
for item in items:
    print(f"  {item.datetime.strftime('%Y-%m-%d')} — {item.properties['eo:cloud_cover']:.1f}% clouds")

# Use the most recent one
item = items[0]
print(f"\nUsing: {item.id}")

## 2. Loading and Displaying Satellite Imagery

Sentinel-2 bands we'll use:
- **B02** (Blue), **B03** (Green), **B04** (Red) — for true color images
- **B08** (Near-Infrared) — for vegetation analysis
- **B11** (SWIR) — for moisture and cloud detection

In [ ]:
def load_band(item, band_name, bbox):
    """Load a single band from a Sentinel-2 item, cropped to bbox."""
    href = item.assets[band_name].href
    with rasterio.open(href) as src:
        projected_bbox = transform_bounds("EPSG:4326", src.crs, *bbox)
        window = from_bounds(*projected_bbox, src.transform)
        data = src.read(1, window=window).astype(float)
        transform = src.window_transform(window)
    return data, transform

def normalize_band(band, percentile=98):
    """Normalize band values to 0-1 range for display."""
    vmin, vmax = np.percentile(band[band > 0], [2, percentile])
    return np.clip((band - vmin) / (vmax - vmin), 0, 1)

# Load RGB + NIR bands
print("Loading bands (this takes ~30s, downloading from cloud)...")
red, transform = load_band(item, "B04", bbox)
green, _ = load_band(item, "B03", bbox)
blue, _ = load_band(item, "B02", bbox)
nir, _ = load_band(item, "B08", bbox)
print(f"Image size: {red.shape[0]}x{red.shape[1]} pixels")

rgb = np.stack([normalize_band(red), normalize_band(green), normalize_band(blue)], axis=-1)
false_color = np.stack([normalize_band(nir), normalize_band(red), normalize_band(green)], axis=-1)

# Display
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

axes[0].imshow(rgb)
axes[0].set_title(f"True Color — {location_name}", fontsize=14, fontweight='bold')
axes[0].axis("off")

axes[1].imshow(false_color)
axes[1].set_title("False Color (NIR-R-G) — vegetation = red", fontsize=14, fontweight='bold')
axes[1].axis("off")

axes[2].imshow(nir, cmap="RdYlGn")
axes[2].set_title("Near-Infrared (B08)", fontsize=14, fontweight='bold')
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 3. NDVI — Vegetation Health Index

NDVI (Normalized Difference Vegetation Index) is the most widely used satellite index. Healthy vegetation reflects near-infrared light strongly.

**Formula:** `NDVI = (NIR - Red) / (NIR + Red)`

Values range from -1 to 1:
- **> 0.6** — Dense, healthy vegetation
- **0.2 - 0.6** — Moderate vegetation
- **< 0.2** — Bare soil, water, or urban areas

In [ ]:
# Calculate NDVI
ndvi = (nir - red) / (nir + red + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(rgb)
axes[0].set_title("True Color", fontsize=14, fontweight='bold')
axes[0].axis("off")

im = axes[1].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title("NDVI — Vegetation Health", fontsize=14, fontweight='bold')
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046, label="NDVI")

plt.tight_layout()
plt.show()

print(f"NDVI stats:")
print(f"  Mean: {ndvi.mean():.3f}")
print(f"  Vegetation pixels (NDVI > 0.3): {(ndvi > 0.3).sum()} / {ndvi.size} ({100*(ndvi > 0.3).mean():.1f}%)")
print(f"  Water/urban pixels (NDVI < 0.1): {(ndvi < 0.1).sum()} / {ndvi.size} ({100*(ndvi < 0.1).mean():.1f}%)")

## 4. Change Detection

Compare two dates to detect changes — new construction, deforestation, flood damage, crop harvesting, etc. This is one of the most powerful capabilities of satellite data.

In [ ]:
# Search for an earlier image of the same area
search_earlier = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-01-01/2025-03-01",
    query={"eo:cloud_cover": {"lt": 15}},
    max_items=1,
)
earlier_items = list(search_earlier.items())

if earlier_items:
    item_earlier = earlier_items[0]
    print(f"Earlier image: {item_earlier.datetime.strftime('%Y-%m-%d')}")
    print(f"Later image:   {item.datetime.strftime('%Y-%m-%d')}")
    print("Loading earlier bands...")

    red_e, _ = load_band(item_earlier, "B04", bbox)
    nir_e, _ = load_band(item_earlier, "B08", bbox)
    green_e, _ = load_band(item_earlier, "B03", bbox)
    blue_e, _ = load_band(item_earlier, "B02", bbox)

    ndvi_earlier = (nir_e - red_e) / (nir_e + red_e + 1e-10)
    ndvi_diff = ndvi - ndvi_earlier

    rgb_earlier = np.stack([normalize_band(red_e), normalize_band(green_e), normalize_band(blue_e)], axis=-1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))

    axes[0].imshow(rgb_earlier)
    axes[0].set_title(f"Earlier — {item_earlier.datetime.strftime('%Y-%m-%d')}", fontsize=14, fontweight='bold')
    axes[0].axis("off")

    axes[1].imshow(rgb)
    axes[1].set_title(f"Later — {item.datetime.strftime('%Y-%m-%d')}", fontsize=14, fontweight='bold')
    axes[1].axis("off")

    im = axes[2].imshow(ndvi_diff, cmap="RdBu", vmin=-0.5, vmax=0.5)
    axes[2].set_title("NDVI Change (blue = more green, red = less)", fontsize=14, fontweight='bold')
    axes[2].axis("off")
    plt.colorbar(im, ax=axes[2], fraction=0.046, label="\u0394NDVI")

    plt.tight_layout()
    plt.show()

    sig_decrease = (ndvi_diff < -0.2).sum()
    sig_increase = (ndvi_diff > 0.2).sum()
    print(f"\nSignificant vegetation loss: {sig_decrease} pixels ({100*sig_decrease/ndvi_diff.size:.1f}%)")
    print(f"Significant vegetation gain: {sig_increase} pixels ({100*sig_increase/ndvi_diff.size:.1f}%)")
else:
    print("No earlier image found — try a different date range")

## 5. The Full Loop — Flood Detection and Analysis

This is the core of the challenge. Your agent should connect real-world events to satellite imagery and produce actual analysis. Below is a complete working example: we analyze the **N'Djamena floods (Oct 2024)** using before/after satellite imagery, NDVI (vegetation loss), and NDWI (water extent).

**This is what your agent should do autonomously.**

In [ ]:
# ============================================================
# FULL LOOP: Flood event → Before/after imagery → NDVI + NDWI
# ============================================================
# N'Djamena, Chad — catastrophic flooding Oct 2024
# Logone & Chari rivers burst banks, 1.9M affected, worst in decades

# STEP 1: WATCH — Identify the event
print("=" * 60)
print("STEP 1: WATCH — Event detected")
print("=" * 60)
print("Event:    N'Djamena floods, Chad")
print("Date:     October 2024 (worst floods in decades)")
print("Impact:   1.9M affected, 600+ dead, massive displacement")
print("Area:     Logone & Chari river basins around N'Djamena")
print()
print("In a real agent, this would come from a news API, disaster")
print("feed, or social media monitor. Here we use a known event")
print("to demonstrate the full analysis pipeline.")

# STEP 2: NAVIGATE — Find before and after satellite imagery
print()
print("=" * 60)
print("STEP 2: NAVIGATE — Finding before/after satellite imagery")
print("=" * 60)

flood_bbox = [14.8, 11.8, 15.4, 12.3]

# BEFORE: Jun 2024 — pre-flood, dry season
search_before = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=flood_bbox,
    datetime="2024-05-01/2024-06-30",
    query={"eo:cloud_cover": {"lt": 15}},
    max_items=5,
)
before_item = list(search_before.items())[0]
print(f"BEFORE: {before_item.datetime.strftime('%Y-%m-%d')} — {before_item.properties['eo:cloud_cover']:.1f}% clouds")

# AFTER: Nov 2024 — still flooded, rivers haven't receded
search_after = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=flood_bbox,
    datetime="2024-10-15/2024-11-15",
    query={"eo:cloud_cover": {"lt": 15}},
    max_items=5,
)
after_item = list(search_after.items())[0]
print(f"AFTER:  {after_item.datetime.strftime('%Y-%m-%d')} — {after_item.properties['eo:cloud_cover']:.1f}% clouds")

# STEP 3: ANALYZE — Load imagery and compute NDVI + NDWI
print()
print("=" * 60)
print("STEP 3: ANALYZE — Loading imagery and computing indices")
print("=" * 60)

print("Downloading BEFORE bands...")
b_red, _ = load_band(before_item, "B04", flood_bbox)
b_green, _ = load_band(before_item, "B03", flood_bbox)
b_blue, _ = load_band(before_item, "B02", flood_bbox)
b_nir, _ = load_band(before_item, "B08", flood_bbox)

print("Downloading AFTER bands...")
a_red, _ = load_band(after_item, "B04", flood_bbox)
a_green, _ = load_band(after_item, "B03", flood_bbox)
a_blue, _ = load_band(after_item, "B02", flood_bbox)
a_nir, _ = load_band(after_item, "B08", flood_bbox)

# Align sizes (tiles can differ by a few pixels)
h = min(b_red.shape[0], a_red.shape[0])
w = min(b_red.shape[1], a_red.shape[1])
b_red, b_green, b_blue, b_nir = b_red[:h,:w], b_green[:h,:w], b_blue[:h,:w], b_nir[:h,:w]
a_red, a_green, a_blue, a_nir = a_red[:h,:w], a_green[:h,:w], a_blue[:h,:w], a_nir[:h,:w]

# Valid data mask — exclude nodata pixels at tile edges
valid = (b_red > 0) & (b_nir > 0) & (b_green > 0) & (a_red > 0) & (a_nir > 0) & (a_green > 0)
print(f"Image size: {h}x{w} pixels — {100*valid.mean():.1f}% valid")

# NDVI: vegetation health (high = healthy, low = bare/water)
b_ndvi = np.where(valid, (b_nir - b_red) / (b_nir + b_red + 1e-10), np.nan)
a_ndvi = np.where(valid, (a_nir - a_red) / (a_nir + a_red + 1e-10), np.nan)
ndvi_diff = a_ndvi - b_ndvi

# NDWI: water detection (positive = water, negative = dry land)
b_ndwi = np.where(valid, (b_green - b_nir) / (b_green + b_nir + 1e-10), np.nan)
a_ndwi = np.where(valid, (a_green - a_nir) / (a_green + a_nir + 1e-10), np.nan)
ndwi_diff = a_ndwi - b_ndwi

# STEP 4: DELIVER — Visualize and report
print()
print("=" * 60)
print("STEP 4: DELIVER — Results")
print("=" * 60)

b_rgb = np.stack([normalize_band(b_red), normalize_band(b_green), normalize_band(b_blue)], axis=-1)
a_rgb = np.stack([normalize_band(a_red), normalize_band(a_green), normalize_band(a_blue)], axis=-1)

# Row 1: True color before/after + vegetation change
fig, axes = plt.subplots(2, 3, figsize=(18, 13))

axes[0,0].imshow(b_rgb)
axes[0,0].set_title(f"BEFORE — {before_item.datetime.strftime('%Y-%m-%d')}", fontsize=13, fontweight='bold')
axes[0,0].axis("off")

axes[0,1].imshow(a_rgb)
axes[0,1].set_title(f"AFTER — {after_item.datetime.strftime('%Y-%m-%d')}", fontsize=13, fontweight='bold')
axes[0,1].axis("off")

im1 = axes[0,2].imshow(ndvi_diff, cmap="RdBu", vmin=-0.5, vmax=0.5)
axes[0,2].set_title("ΔNDVI — Vegetation Change\n(red = destroyed, blue = growth)", fontsize=12, fontweight='bold')
axes[0,2].axis("off")
plt.colorbar(im1, ax=axes[0,2], fraction=0.046, label="ΔNDVI")

# Row 2: NDWI before, after, and change
axes[1,0].imshow(b_ndwi, cmap="RdYlBu", vmin=-0.5, vmax=0.5)
axes[1,0].set_title("NDWI Before — Water Extent", fontsize=13, fontweight='bold')
axes[1,0].axis("off")

axes[1,1].imshow(a_ndwi, cmap="RdYlBu", vmin=-0.5, vmax=0.5)
axes[1,1].set_title("NDWI After — Water Extent", fontsize=13, fontweight='bold')
axes[1,1].axis("off")

im2 = axes[1,2].imshow(ndwi_diff, cmap="RdYlBu", vmin=-0.3, vmax=0.3)
axes[1,2].set_title("ΔNDWI — Flood Extent\n(blue = new water)", fontsize=12, fontweight='bold')
axes[1,2].axis("off")
plt.colorbar(im2, ax=axes[1,2], fraction=0.046, label="ΔNDWI")

plt.suptitle("N'Djamena Floods (Oct 2024) — Satellite Analysis", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Metrics (using only valid pixels)
water_before = 100 * np.nanmean(b_ndwi > 0)
water_after = 100 * np.nanmean(a_ndwi > 0)
veg_loss = 100 * np.nanmean(ndvi_diff < -0.15)
new_flood = 100 * np.nanmean(ndwi_diff > 0.15)

print(f"\n{'─' * 50}")
print(f"  FLOOD IMPACT REPORT")
print(f"{'─' * 50}")
print(f"  Water coverage before:  {water_before:.1f}%")
print(f"  Water coverage after:   {water_after:.1f}%")
print(f"  New flooding:           +{water_after - water_before:.1f} percentage points")
print(f"  Vegetation destroyed:   {veg_loss:.1f}% of area")
print(f"  Area with new water:    {new_flood:.1f}%")
print(f"{'─' * 50}")
print(f"\n--- YOUR AGENT IMPROVES FROM HERE ---")
print(f"Ideas: Cross-reference with OpenStreetMap to estimate")
print(f"affected buildings/roads, track flood recession over weeks,")
print(f"combine with population data for displacement estimates.")

## 6. Automated Image Quality Assessment

Before your agent can analyze anything, it needs to pick the RIGHT image. Cloud cover metadata from the catalog is approximate — the actual image might have clouds right over your area of interest.

Sentinel-2 includes a **Scene Classification Layer (SCL)** — an ML model that classifies every pixel into: vegetation, water, built-up/urban, cloud, cloud shadow, snow, etc. Your agent can use this to:
1. **Score every candidate image** for actual usability over your specific AOI
2. **Pick the best one automatically** — comparing clear summer vs cloudy winter images
3. **Report what's in the scene** — vegetation, urban infrastructure, water bodies, cloud contamination

This is critical for autonomous agents — they need to make data quality decisions without human review.

In [ ]:
# ============================================================
# AUTOMATED IMAGE QUALITY ASSESSMENT
# Your agent needs to pick the best image — here's how
# ============================================================

# Sentinel-2 Scene Classification Layer (SCL) — ML-based pixel classification
# Note: Class 5 is "Not Vegetated" which includes urban/built-up AND bare soil
SCL_CLASSES = {
    0: ("No Data",         "#000000"),
    1: ("Saturated",       "#ff0000"),
    2: ("Dark/Shadow",     "#404040"),
    3: ("Cloud Shadow",    "#8B4513"),
    4: ("Vegetation",      "#228B22"),
    5: ("Built-up / Soil", "#DAA520"),   # Urban, roads, buildings, bare soil
    6: ("Water",           "#0066CC"),
    7: ("Cloud (Low)",     "#C0C0C0"),
    8: ("Cloud (Medium)",  "#808080"),
    9: ("Cloud (High)",    "#FFFFFF"),
    10: ("Thin Cirrus",    "#E0E0FF"),
    11: ("Snow/Ice",       "#00FFFF"),
}

# Categories for quality scoring
USABLE = {4, 5, 6}          # Vegetation, Built-up/Soil, Water — good data
CLOUDY = {7, 8, 9, 10}      # Any cloud type — bad
SHADOW = {2, 3}              # Dark areas and cloud shadows — degraded

def assess_image_quality(item, bbox):
    """Load SCL band and compute quality metrics for a Sentinel-2 image."""
    try:
        href = item.assets["SCL"].href
        with rasterio.open(href) as src:
            projected_bbox = transform_bounds("EPSG:4326", src.crs, *bbox)
            window = from_bounds(*projected_bbox, src.transform)
            scl = src.read(1, window=window)
    except Exception as e:
        return None

    total = scl.size
    if total == 0:
        return None

    # Count each class
    counts = {}
    for val, (name, _) in SCL_CLASSES.items():
        counts[val] = int((scl == val).sum())

    # Quality metrics
    usable_pct = 100 * sum(counts.get(v, 0) for v in USABLE) / total
    cloud_pct = 100 * sum(counts.get(v, 0) for v in CLOUDY) / total
    shadow_pct = 100 * sum(counts.get(v, 0) for v in SHADOW) / total
    veg_pct = 100 * counts.get(4, 0) / total
    water_pct = 100 * counts.get(6, 0) / total
    buildup_pct = 100 * counts.get(5, 0) / total

    # Quality score: usable area minus penalties
    score = usable_pct - (cloud_pct * 1.5) - (shadow_pct * 0.5)

    return {
        "date": item.datetime.strftime('%Y-%m-%d'),
        "catalog_cloud": item.properties['eo:cloud_cover'],
        "usable_pct": usable_pct,
        "cloud_pct": cloud_pct,
        "shadow_pct": shadow_pct,
        "veg_pct": veg_pct,
        "water_pct": water_pct,
        "buildup_pct": buildup_pct,
        "score": score,
        "scl": scl,
        "counts": counts,
        "item": item,
    }

# ============================================================
# SEARCH: Clear, partially cloudy, AND heavily cloudy images
# ============================================================
print("=" * 60)
print("  AUTOMATED IMAGE QUALITY ASSESSMENT")
print(f"  Location: {location_name}")
print("=" * 60)

# Clear summer images (already have from Section 1)
# Partially cloudy — search broader range for 15-60% catalog cloud
search_partial = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2024-09-01/2025-06-01",
    query={"eo:cloud_cover": {"lt": 60, "gt": 15}},
    max_items=10,
)
partial_items = list(search_partial.items())

# Heavily cloudy — 70%+ cloud
search_heavy = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-01-01/2025-03-31",
    query={"eo:cloud_cover": {"lt": 95, "gt": 70}},
    max_items=5,
)
heavy_items = list(search_heavy.items())

# Assess all partial candidates to find one with 20-60% ACTUAL cloud over AOI
print(f"\n  Scanning {len(partial_items)} partially cloudy candidates...")
partial_results = []
for pi in partial_items:
    r = assess_image_quality(pi, bbox)
    if r and 15 < r['cloud_pct'] < 70:
        partial_results.append(r)
        print(f"    {r['date']}: actual {r['cloud_pct']:.0f}% cloud over AOI (catalog: {r['catalog_cloud']:.0f}%)")

# Assess heavy cloud candidates for the worst
print(f"\n  Scanning {len(heavy_items)} heavily cloudy candidates...")
heavy_results = []
for hi in heavy_items:
    r = assess_image_quality(hi, bbox)
    if r and r['cloud_pct'] > 70:
        heavy_results.append(r)
        print(f"    {r['date']}: actual {r['cloud_pct']:.0f}% cloud over AOI (catalog: {r['catalog_cloud']:.0f}%)")

# Pick the best clear image from Section 1
clear_results = []
for ci in items[:3]:  # Top 3 clear
    r = assess_image_quality(ci, bbox)
    if r:
        clear_results.append(r)

# Select our three showcase images
best_clear = sorted(clear_results, key=lambda x: x['score'], reverse=True)[0]
best_partial = sorted(partial_results, key=lambda x: x['cloud_pct'])[len(partial_results)//2] if partial_results else None  # Pick middle one
worst_heavy = sorted(heavy_results, key=lambda x: x['cloud_pct'], reverse=True)[0] if heavy_results else None

# Combine ALL results for the ranking table
all_results = clear_results + partial_results + heavy_results
all_results.sort(key=lambda x: x['score'], reverse=True)

# ============================================================
# VISUALIZE: Best / Partial Cloud / Worst — 3 rows
# ============================================================
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

scl_cmap = ListedColormap([
    '#000000', '#ff0000', '#404040', '#8B4513', '#228B22',
    '#DAA520', '#0066CC', '#C0C0C0', '#808080', '#FFFFFF',
    '#E0E0FF', '#00FFFF'
])

showcase = []
if best_clear:
    showcase.append((best_clear, "BEST — Clear", "green"))
if best_partial:
    showcase.append((best_partial, "PARTIAL CLOUD", "#CC8800"))
if worst_heavy:
    showcase.append((worst_heavy, "WORST — Overcast", "red"))

n_rows = len(showcase)
fig, axes = plt.subplots(n_rows, 2, figsize=(14, 6 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]

for idx, (result, row_label, color) in enumerate(showcase):
    target_item = result['item']
    r, _ = load_band(target_item, "B04", bbox)
    g, _ = load_band(target_item, "B03", bbox)
    b, _ = load_band(target_item, "B02", bbox)
    rgb_img = np.stack([normalize_band(r), normalize_band(g), normalize_band(b)], axis=-1)

    grade = "A+" if result['score'] > 90 else "A" if result['score'] > 80 else "B" if result['score'] > 60 else "C" if result['score'] > 40 else "D"
    axes[idx, 0].imshow(rgb_img)
    axes[idx, 0].set_title(f"{row_label}: {result['date']} — Score {result['score']:.0f} [{grade}]",
                            fontsize=12, fontweight='bold', color=color)
    axes[idx, 0].axis("off")

    im = axes[idx, 1].imshow(result['scl'], cmap=scl_cmap, vmin=0, vmax=11, interpolation='nearest')
    axes[idx, 1].set_title(f"SCL — {result['usable_pct']:.0f}% usable, {result['cloud_pct']:.0f}% cloud, {result['shadow_pct']:.0f}% shadow",
                            fontsize=11, fontweight='bold')
    axes[idx, 1].axis("off")

# SCL Legend
legend_items = [Patch(facecolor=SCL_CLASSES[k][1], edgecolor='gray', label=SCL_CLASSES[k][0])
                for k in [4, 5, 6, 7, 8, 9, 3, 2]]
fig.legend(handles=legend_items, loc='lower center', ncol=4, fontsize=10,
           bbox_to_anchor=(0.5, -0.02), frameon=False)

plt.suptitle("Image Quality Assessment — Clear vs Partial Cloud vs Overcast",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================================
# DETAILED REPORT
# ============================================================
print(f"\n{'=' * 60}")
print(f"  QUALITY RANKING — All {len(all_results)} candidates assessed")
print(f"{'=' * 60}")
print(f"  {'Rank':<5} {'Date':<12} {'Score':<8} {'Usable':<9} {'Cloud':<9} {'Shadow':<9} {'Vegetation':<11} {'Built-up':<10} {'Water':<8} {'Decision'}")
print(f"  {'─'*5} {'─'*12} {'─'*8} {'─'*9} {'─'*9} {'─'*9} {'─'*11} {'─'*10} {'─'*8} {'─'*12}")

for i, r in enumerate(all_results):
    decision = "✓ USE THIS" if i == 0 else "backup" if r['score'] > 70 else "⚠ partial" if r['score'] > 20 else "✗ skip"
    print(f"  {i+1:<5} {r['date']:<12} {r['score']:<8.0f} {r['usable_pct']:<9.1f} {r['cloud_pct']:<9.1f} {r['shadow_pct']:<9.1f} {r['veg_pct']:<11.1f} {r['buildup_pct']:<10.1f} {r['water_pct']:<8.1f} {decision}")

print(f"\n  Agent recommendation: Use {best_clear['date']}")
print(f"    → {best_clear['usable_pct']:.0f}% usable — {best_clear['veg_pct']:.0f}% vegetation, {best_clear['buildup_pct']:.0f}% built-up/urban, {best_clear['water_pct']:.0f}% water")
print(f"    → Catalog: {best_clear['catalog_cloud']:.0f}% cloud → Actual over AOI: {best_clear['cloud_pct']:.0f}%")

if best_partial:
    print(f"\n  Partial cloud example: {best_partial['date']}")
    print(f"    → {best_partial['cloud_pct']:.0f}% of AOI obscured by cloud — analysis possible but degraded")
    print(f"    → An agent could mask cloudy pixels and analyze the {best_partial['usable_pct']:.0f}% that's usable")

if worst_heavy:
    print(f"\n  Worst example: {worst_heavy['date']}")
    print(f"    → {worst_heavy['cloud_pct']:.0f}% cloud — image is effectively useless")
    print(f"    → An agent must reject this and search for alternatives")

print(f"\n  Your agent should do this automatically for every image it considers.")

## 7. Useful Helper Functions

Copy these into your agent code.

In [ ]:
def search_sentinel2(lat, lon, start_date, end_date, max_cloud=20, buffer=0.05):
    """Search for Sentinel-2 images near a lat/lon point."""
    bbox = [lon - buffer, lat - buffer, lon + buffer, lat + buffer]
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=f"{start_date}/{end_date}",
        query={"eo:cloud_cover": {"lt": max_cloud}},
        max_items=10,
    )
    return list(search.items())

def load_rgb(item, bbox):
    """Load true-color RGB image from a Sentinel-2 item."""
    red, transform = load_band(item, "B04", bbox)
    green, _ = load_band(item, "B03", bbox)
    blue, _ = load_band(item, "B02", bbox)
    rgb = np.stack([normalize_band(red), normalize_band(green), normalize_band(blue)], axis=-1)
    return rgb

def calculate_ndvi(item, bbox):
    """Calculate NDVI for a Sentinel-2 item."""
    red, _ = load_band(item, "B04", bbox)
    nir, _ = load_band(item, "B08", bbox)
    return (nir - red) / (nir + red + 1e-10)

def detect_water(item, bbox):
    """Simple water detection using NDWI (Normalized Difference Water Index)."""
    green, _ = load_band(item, "B03", bbox)
    nir, _ = load_band(item, "B08", bbox)
    ndwi = (green - nir) / (green + nir + 1e-10)
    return ndwi > 0.3  # Boolean mask: True = water

def detect_burn_scars(item, bbox):
    """Detect burned areas using NBR (Normalized Burn Ratio)."""
    nir, _ = load_band(item, "B08", bbox)
    swir, _ = load_band(item, "B12", bbox)
    nbr = (nir - swir) / (nir + swir + 1e-10)
    return nbr < -0.1  # Boolean mask: True = burned

# Example usage
print("Helper functions loaded!")
print("  search_sentinel2(lat, lon, start, end) — find images")
print("  load_rgb(item, bbox) — get true color image")
print("  calculate_ndvi(item, bbox) — vegetation health")
print("  detect_water(item, bbox) — water body mask")
print("  detect_burn_scars(item, bbox) — fire damage mask")

## 7. What's Next?

You now have the building blocks. Here's how to turn this into an Earth Intelligence Agent:

**1. Choose a use case** — What real-world problem will your agent solve? Who's the customer?

**2. Add data sources** — The more sources your agent fuses, the more powerful it becomes.

**3. Build the loop** — Your agent should:
- **Watch** real-world data streams for triggers
- **Navigate** to relevant satellite imagery automatically
- **Analyze** with tools it builds or adapts itself
- **Deliver** actionable intelligence to a customer
- **Improve** based on results

**4. Make it autonomous** — The best agents don't need hand-holding. They detect, analyze, and deliver on their own.

**5. Make it better over time** — Can your agent learn from its own results? Can it discover what works and do more of it?

### Free APIs that work with no API key:
| Source | What | URL |
|--------|------|-----|
| **Planetary Computer** | Sentinel-2 satellite imagery | planetarycomputer.microsoft.com |
| **USGS Earthquakes** | Real-time global earthquakes | earthquake.usgs.gov |
| **Open-Meteo** | Weather forecasts + historical | open-meteo.com |
| **OpenStreetMap / Overpass** | Roads, buildings, land use | overpass-api.de |
| **Nominatim** | Geocoding (place name → coordinates) | nominatim.openstreetmap.org |

### Free APIs that need a key (free tier):
| Source | What | URL |
|--------|------|-----|
| **NewsAPI** | Headlines from 80k+ sources | newsapi.org (100 req/day free) |
| **OpenWeatherMap** | Current + forecast weather | openweathermap.org (1000 req/day free) |
| **Alpha Vantage** | Stocks, forex, crypto | alphavantage.co (25 req/day free) |

### No-key Python libraries:
- `yfinance` — stock and commodity data (no key needed)
- `geopy` — geocoding
- `folium` — interactive maps
- `openai` / `anthropic` — LLM APIs for your agent's reasoning
- `scikit-learn` — ML classification and clustering

Good luck! Build something that discovers what nobody else has thought to look for.